# LAB 08 — Pretrained Transformers and BERT Fine-Tuning
**Môn học:** Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Giảng viên / TA:** Phạm Ngọc Hải  
**Sinh viên thực hiện:** Đỗ Khắc Đức Thành - MSSV: 24001697  
**Chủ đề:** From Training a Transformer to Using a Pretrained Language Model  

---
### Mục tiêu chính:
1. Triển khai mô hình phân loại văn bản chuẩn BERT-style: `Pretrained Transformer -> [CLS] representation -> Linear layer -> Softmax`.
2. Thực hiện **Experiment 1**: So sánh Transformer Train from Scratch vs Pretrained Fine-Tuning.
3. Thực hiện **Experiment 2**: So sánh Feature Extraction (Frozen Encoder) vs Full Fine-Tuning.
4. Thực hiện **Experiment 3**: Đánh giá hiệu quả dữ liệu (Data Efficiency: 10%, 25%, 50%, 100%).
5. Thực hiện **Experiment 4**: Khảo sát Maximum Sequence Length $T \in \{64, 128, 256\}$ và liên hệ độ phức tạp $O(T^2)$.
6. Thực hiện **Experiment 5**: Khảo sát hiện tượng Domain Shift (In-domain vs Out-of-domain).
7. Phân tích định lượng, vẽ Confusion Matrix, Data Efficiency curve và Confidence Analysis.

## 1. Thiết lập Môi trường và Khởi tạo Siêu Tham Số

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig

# Đảm bảo tính tái lập (Reproducibility - Mục 23)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Thiết bị tính toán: {device}")
print(f"[*] PyTorch Version: {torch.__version__}")

## 2. Xây dựng Kiến Trúc Mô Hình (Section 13 — Classification Head Pipeline)
Pipeline:
```text
Input text -> Tokenizer -> Input IDs -> Pretrained Transformer
           -> Contextual representations -> [CLS] representation
           -> Linear layer -> Class logits -> Softmax -> Prediction
```

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_classes=2, freeze_encoder=False, from_scratch=False):
        super(TransformerClassifier, self).__init__()
        self.from_scratch = from_scratch
        
        if from_scratch:
            # Khởi tạo Transformer ngẫu nhiên (Model A - Train from Scratch)
            config = AutoConfig.from_pretrained(model_name)
            config.num_hidden_layers = 4
            config.num_attention_heads = 4
            config.hidden_size = 256
            config.intermediate_size = 512
            self.encoder = AutoModel.from_config(config)
            hidden_size = config.hidden_size
        else:
            # Tải Pretrained Transformer (Model B - Pretrained)
            self.encoder = AutoModel.from_pretrained(model_name)
            hidden_size = self.encoder.config.hidden_size
            
            if freeze_encoder:
                # Đóng băng tham số encoder (Mode A - Frozen Encoder)
                for param in self.encoder.parameters():
                    param.requires_grad = False
                    
        self.dropout = nn.Dropout(0.1)
        # Classification Head: thành phần task-specific
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Trích xuất vector biểu diễn [CLS] tại vị trí index 0
        cls_rep = outputs.last_hidden_state[:, 0, :]
        cls_rep = self.dropout(cls_rep)
        logits = self.classifier(cls_rep)
        return logits

## 3. Chuẩn bị Dữ liệu IMDb và Out-of-Domain Dataset

In [ ]:
from pretrained_classification import build_benchmark_datasets, TextSentimentDataset, train_and_evaluate

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_data, val_data, test_data, ood_data = build_benchmark_datasets()

print(f"- Số lượng mẫu Train (Domain A - IMDb): {len(train_data[0])}")
print(f"- Số lượng mẫu Val: {len(val_data[0])}")
print(f"- Số lượng mẫu Test (In-domain): {len(test_data[0])}")
print(f"- Số lượng mẫu Test (Out-of-domain - News/Tech): {len(ood_data[0])}")

## 4. Chạy Toàn Bộ Thí Nghiệm và Thu Thập Kết Quả
Chạy hàm `run_all_experiments()` để tự động thực hiện 5 thực nghiệm, tính toán metrics, xuất `results.csv` và tạo các hình ảnh trực quan.

In [ ]:
from pretrained_classification import run_all_experiments

# Chạy toàn bộ 5 thực nghiệm
run_all_experiments()

## 5. Trực Quan Hóa Kết Quả Định Lượng (Quantitative Evaluation)

In [ ]:
# Hiển thị bảng tổng kết kết quả từ results.csv
df_res = pd.read_csv("results.csv")
df_res

### 5.1. Ma Trận Nhầm Lẫn (Confusion Matrix)

In [ ]:
from IPython.display import Image
Image(filename="confusion_matrix.png")

### 5.2. Đường Cong Hiệu Quả Dữ Liệu (Data Efficiency Curve)

In [ ]:
Image(filename="data_efficiency.png")

### 5.3. Phân Tích Độ Tự Tin Dự Đoán (Confidence Analysis)

In [ ]:
Image(filename="confidence_analysis.png")

## 6. Tổng Kết và Kết Nối Sang LAB 09
- Mô hình **Pretrained BERT Fine-tuning** thể hiện sự áp đảo toàn diện so với Transformer train from scratch về cả độ chính xác, điểm F1 và tính hiệu quả dữ liệu.
- Chế độ **Full Fine-Tuning** vượt trội hơn **Frozen Encoder**, khẳng định vai trò của việc tái định hình attention weights cho bài toán cụ thể.
- Độ dài chuỗi $T=128$ đạt điểm cân bằng tối ưu giữa hiệu năng và độ phức tạp tính toán $O(T^2)$.
- Hiện tượng **Domain Shift** và **Overconfidence** là những hạn chế trọng tâm cần lưu ý khi ứng dụng thực tế.
- **Chuyển giao sang LAB 09:** Mở rộng từ Sequence Classification (nhãn toàn câu qua token `[CLS]`) sang Token Classification (gán nhãn cho từng từ: Named Entity Recognition - NER).